# Font Identifier - Training Notebook

This notebook handles the end-to-end training of the Font Identifier model using EfficientNet-B0. It includes a robust checkpointing system to handle Colab disconnects.

## Step 1 — Verify GPU Access

In [ ]:
!nvidia-smi

## Step 2 — Mount Google Drive
All data and checkpoints will be stored in Drive to survive session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/fontidentifier'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/dataset', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/fonts', exist_ok=True)

print("Drive mounted and folders ready.")

## Step 3 — Clone Repository

In [ ]:
# REPLACE WITH YOUR ACTUAL REPO URL
!git clone https://github.com/YOUR_USERNAME/fontidentifier.git /content/fontidentifier
%cd /content/fontidentifier/model
!ls -la

## Step 4 — Install Dependencies

In [ ]:
!pip install -q timm supabase python-dotenv
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 5 — Set Environment Variables

In [ ]:
import os
os.environ['GOOGLE_FONTS_API_KEY'] = 'your_api_key'
os.environ['SUPABASE_URL']         = 'your_supabase_url'
os.environ['SUPABASE_SERVICE_KEY'] = 'your_service_key'
print("Secrets set (locally in memory).")

## Step 6 — Download Fonts
Running this once will cache all fonts in Google Drive.

In [ ]:
import sys
sys.path.insert(0, '/content/fontidentifier/model')
from scripts.download_fonts import download_google_fonts, download_fontsquirrel

# Override local path to point to Drive
import scripts.download_fonts
scripts.download_fonts.FONTS_DIR = f'{DRIVE_DIR}/fonts'

download_google_fonts()
download_fontsquirrel(limit=2000)

## Step 7 — Generate Training Dataset

In [ ]:
from scripts.generate_dataset import generate_dataset
import scripts.generate_dataset

# Override paths
scripts.generate_dataset.FONTS_DIR = f'{DRIVE_DIR}/fonts'
scripts.generate_dataset.OUTPUT_DIR = f'{DRIVE_DIR}/dataset'

generate_dataset()

## Step 8 — Start Training
This loop includes the explicit auto-resume logic.

In [ ]:
import train
train.DATASET_DIR = f'{DRIVE_DIR}/dataset'
train.DRIVE_DIR = f'{DRIVE_DIR}/checkpoints'
train.CHECKPOINT_PATH = os.path.join(train.DRIVE_DIR, "latest.pt")
train.BEST_MODEL_PATH = os.path.join(train.DRIVE_DIR, "best_model.pt")

train.train()

## Step 9 — Export Final Model

In [ ]:
import torch
from train import FontNet
import json

device = torch.device('cuda')
# Load best checkpoint to get num_classes
best_ckpt = torch.load(train.BEST_MODEL_PATH, map_location=device)
num_classes = best_ckpt['num_classes']

model = FontNet(num_classes=num_classes).to(device)
model.load_state_dict(best_ckpt['model_state'])
model.eval()

model_cpu = model.cpu()
model_cpu.eval()

scripted = torch.jit.script(model_cpu)
scripted.save(f'{DRIVE_DIR}/font_model_scripted.pt')
print(f"Final model saved to {DRIVE_DIR}/font_model_scripted.pt")